# Exercise 05 Solution — BCM Plasticity on GPU

In [ ]:
!nvidia-smi

In [ ]:
%%writefile bcm_solution.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset;
__constant__ float c_tau_E, c_E_E;
__constant__ int   c_T_ref;
__constant__ float c_tau_r, c_tau_theta, c_eta, c_w_max, c_w_min;

// PART 1 SOLUTION: Smoothed rate + BCM threshold update
__global__ void update_rates_and_theta(
    const int* fired, float* r, float* theta, int N
) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= N) return;

    // Exponential decay of rate estimate, then add instantaneous rate
    // fired[j] is 0 or 1; dividing by (dt/1000) converts to Hz
    r[j] = r[j] * expf(-c_dt / c_tau_r) + fired[j] / (c_dt * 0.001f);

    // Euler step for sliding threshold toward r^2
    theta[j] += (c_dt / c_tau_theta) * (r[j] * r[j] - theta[j]);
    if (theta[j] < 0.1f) theta[j] = 0.1f;  // prevent theta from going to zero
}

// LIF step (unchanged)
__global__ void lif_step(
    float* V, float* g_E, int* ref, int* fired, const float* I_ext, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    g_E[i] *= expf(-c_dt / c_tau_E);
    fired[i] = 0;
    if (ref[i] > 0) { ref[i]--; V[i] = c_V_reset; return; }
    float I_syn = -g_E[i] * (V[i] - c_E_E);
    V[i] += c_dt / c_tau_m * (-(V[i] - c_E_L) + c_Rm * (I_ext[i] + I_syn));
    if (V[i] >= c_V_th) { V[i] = c_V_reset; ref[i] = c_T_ref; fired[i] = 1; }
}

// PART 3 SOLUTION: BCM weight update
__global__ void bcm_update(
    const int* fired, const int* row_ptr, const int* col_idx,
    float* weights, const float* r, const float* theta, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N || !fired[i]) return;

    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++) {
        int j = col_idx[k];

        // BCM rule: dw = eta * r_pre * r_post * (r_post - theta)
        float dw = c_eta * r[i] * r[j] * (r[j] - theta[j]);

        // Soft weight bounds
        if (dw > 0.f) dw *= (c_w_max - weights[k]) / c_w_max;
        else          dw *= (weights[k] - c_w_min) / c_w_max;

        atomicAdd(&weights[k], dw);
    }
}

// Propagation (unchanged)
__global__ void propagate(
    const int* fired, const int* row_ptr, const int* col_idx,
    const float* weights, float* g_E, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N || !fired[i]) return;
    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++)
        atomicAdd(&g_E[col_idx[k]], weights[k]);
}

void build_csr(int N, float p, float w_init,
               int** rp, int** ci, float** wv, int* nnz_out)
{
    srand(42);
    int* cnt=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i!=j && (float)rand()/RAND_MAX < p) cnt[i]++;
    *rp=(int*)malloc((N+1)*sizeof(int)); (*rp)[0]=0;
    for(int i=0;i<N;i++) (*rp)[i+1]=(*rp)[i]+cnt[i];
    int nnz=(*rp)[N]; *nnz_out=nnz;
    *ci=(int*)malloc(nnz*sizeof(int)); *wv=(float*)malloc(nnz*sizeof(float));
    srand(42); int* pos=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i!=j && (float)rand()/RAND_MAX < p) {
            int k=(*rp)[i]+pos[i]++; (*ci)[k]=j; (*wv)[k]=w_init; }
    free(cnt); free(pos);
}

int main(int argc, char** argv)
{
    int   N    = (argc>1) ? atoi(argv[1]) : 200;
    float T_ms = (argc>2) ? atof(argv[2]) : 3000.f;
    float dt   = 0.1f;
    int   T    = (int)(T_ms / dt);

    float tau_m=20.f,E_L=-65.f,Rm=10.f,V_th=-55.f,V_reset=-70.f;
    int T_ref=(int)(2.f/dt);
    float tau_E=5.f, E_E=0.f;
    float tau_r=100.f, tau_theta=1000.f, eta=0.001f;
    float w_max=0.5f, w_min=0.0f, w_init=0.15f, p_conn=0.2f;

    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,       &dt,       sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,    &tau_m,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,      &E_L,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,       &Rm,       sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,     &V_th,     sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset,  &V_reset,  sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,    &T_ref,    sizeof(int)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_E,    &tau_E,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_E,      &E_E,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_r,    &tau_r,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_theta,&tau_theta,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_eta,      &eta,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_w_max,    &w_max,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_w_min,    &w_min,    sizeof(float)));

    int *h_rp,*h_ci; float *h_wv; int nnz;
    build_csr(N,p_conn,w_init,&h_rp,&h_ci,&h_wv,&nnz);

    float *d_V,*d_gE,*d_Iext,*d_r,*d_theta,*d_wv;
    int   *d_ref,*d_fired,*d_rp,*d_ci;
    CUDA_CHECK(cudaMalloc(&d_V,    N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_gE,   N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_Iext, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_r,    N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_theta,N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_ref,  N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_fired,N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_wv,   nnz*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_rp,  (N+1)*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_ci,   nnz*sizeof(int)));

    float* h_V=(float*)malloc(N*sizeof(float));
    float* h_Iext=(float*)malloc(N*sizeof(float));
    float* h_theta=(float*)malloc(N*sizeof(float));
    srand(7);
    for(int i=0;i<N;i++) {
        h_V[i]=E_L; h_Iext[i]=1.0f+0.8f*(float)rand()/RAND_MAX;
        h_theta[i]=5.0f;
    }
    CUDA_CHECK(cudaMemcpy(d_V,   h_V,   N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Iext,h_Iext,N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_theta,h_theta,N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_gE,0,N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_r, 0,N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_ref,0,N*sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d_wv,h_wv,nnz*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_rp,h_rp,(N+1)*sizeof(int),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_ci,h_ci,nnz*sizeof(int),cudaMemcpyHostToDevice));

    int thr=256, blk=(N+thr-1)/thr;
    FILE* fsnap=fopen("bcm_weights_sol.txt","w");
    FILE* fspikes=fopen("bcm_spikes_sol.txt","w");
    int* h_fired=(int*)malloc(N*sizeof(int));
    int snap_every=(int)(500/dt);

    cudaEvent_t ev0,ev1; float sim_ms;
    CUDA_CHECK(cudaEventCreate(&ev0)); CUDA_CHECK(cudaEventCreate(&ev1));
    CUDA_CHECK(cudaEventRecord(ev0));

    for(int step=0;step<T;step++) {
        lif_step<<<blk,thr>>>(d_V,d_gE,d_ref,d_fired,d_Iext,N);
        propagate<<<blk,thr>>>(d_fired,d_rp,d_ci,d_wv,d_gE,N);

        // SOLUTION: call the two learning kernels
        update_rates_and_theta<<<blk,thr>>>(d_fired,d_r,d_theta,N);
        if (step > (int)(500/dt))
            bcm_update<<<blk,thr>>>(d_fired,d_rp,d_ci,d_wv,d_r,d_theta,N);

        if(step%10==0) {
            CUDA_CHECK(cudaMemcpy(h_fired,d_fired,N*sizeof(int),cudaMemcpyDeviceToHost));
            float t_ms=step*dt;
            for(int i=0;i<N;i++) if(h_fired[i]) fprintf(fspikes,"%d %.1f\n",i,t_ms);
        }
        if(step%snap_every==0) {
            CUDA_CHECK(cudaMemcpy(h_wv,d_wv,nnz*sizeof(float),cudaMemcpyDeviceToHost));
            fprintf(fsnap,"# t=%.0f ms\n",step*dt);
            for(int k=0;k<nnz;k++) fprintf(fsnap,"%.6f\n",h_wv[k]);
        }
    }

    CUDA_CHECK(cudaEventRecord(ev1)); CUDA_CHECK(cudaEventSynchronize(ev1));
    CUDA_CHECK(cudaEventElapsedTime(&sim_ms,ev0,ev1));
    printf("N=%d, T=%.0f ms, GPU=%.1f ms, speedup=%.1fx\n",
           N, T_ms, sim_ms, T_ms/sim_ms);
    fclose(fsnap); fclose(fspikes);
    CUDA_CHECK(cudaEventDestroy(ev0)); CUDA_CHECK(cudaEventDestroy(ev1));
    cudaFree(d_V);cudaFree(d_gE);cudaFree(d_Iext);
    cudaFree(d_r);cudaFree(d_theta);cudaFree(d_ref);cudaFree(d_fired);
    cudaFree(d_wv);cudaFree(d_rp);cudaFree(d_ci);
    free(h_V);free(h_Iext);free(h_theta);free(h_fired);
    free(h_rp);free(h_ci);free(h_wv);
    return 0;
}

In [ ]:
!nvcc -O2 -o bcm_solution bcm_solution.cu -lm && ./bcm_solution 200 3000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load weight snapshots
snapshots = {}
with open('bcm_weights_sol.txt') as f:
    current_t, current_w = None, []
    for line in f:
        line = line.strip()
        if line.startswith('#'):
            if current_t is not None: snapshots[current_t] = np.array(current_w)
            current_t = float(line.split('=')[1].split()[0]); current_w = []
        else: current_w.append(float(line))
    if current_t is not None: snapshots[current_t] = np.array(current_w)

times = sorted(snapshots.keys())
n_plots = min(len(times), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
cmap = plt.cm.viridis(np.linspace(0, 1, n_plots))

mean_weights = []
for idx, t in enumerate(times[:n_plots]):
    w = snapshots[t]
    mean_weights.append(w.mean())
    axes[idx].hist(w, bins=40, color=cmap[idx], alpha=0.85, edgecolor='white', lw=0.3)
    axes[idx].axvline(w.mean(), color='crimson', linestyle='--', lw=1.5,
                       label=f'μ={w.mean():.3f}\nσ={w.std():.3f}')
    axes[idx].set_title(f't = {t:.0f} ms', fontsize=12)
    axes[idx].set_xlabel('Weight w'); axes[idx].set_ylabel('Count')
    axes[idx].legend(fontsize=9); axes[idx].grid(True, alpha=0.3)
    axes[idx].set_xlim(0, 0.5)

plt.suptitle('BCM Solution: Weight Distribution Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bcm_weight_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean weight trajectory
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times[:n_plots], mean_weights, 'o-', color='teal', lw=2, markersize=6)
ax.set_xlabel('Time (ms)', fontsize=12); ax.set_ylabel('Mean weight', fontsize=12)
ax.set_title('BCM: Mean Synaptic Weight Over Time', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Answer Key

### Part 1 — Rate and threshold update

```c
// Smoothed rate (exponential filter + instantaneous spike contribution)
r[j] = r[j] * expf(-c_dt / c_tau_r) + fired[j] / (c_dt * 0.001f);

// BCM threshold slides toward r^2 (Euler step)
theta[j] += (c_dt / c_tau_theta) * (r[j] * r[j] - theta[j]);
```

**Why `fired[j] / (dt * 0.001)`?** `fired[j]` is 0 or 1 per timestep. Dividing by `dt` in seconds converts "spikes per timestep" to "spikes per second" (Hz). Since `dt` is in ms, `dt * 0.001f` converts to seconds.

### Part 3 — BCM weight update

```c
float dw = c_eta * r[i] * r[j] * (r[j] - theta[j]);  // BCM rule
if (dw > 0.f) dw *= (c_w_max - weights[k]) / c_w_max;  // LTP soft bound
else          dw *= (weights[k] - c_w_min) / c_w_max;   // LTD soft bound
atomicAdd(&weights[k], dw);
```

### Part 4 — Kernel launch calls

```c
update_rates_and_theta<<<blk,thr>>>(d_fired, d_r, d_theta, N);
if (step > (int)(500/dt))
    bcm_update<<<blk,thr>>>(d_fired, d_rp, d_ci, d_wv, d_r, d_theta, N);
```

The 500ms warmup ensures neurons have established a baseline firing rate before plasticity begins — otherwise the very early transient would drive `theta` away from a useful operating point.

### Reflection Answers

1. **Homeostasis:** When a neuron is chronically active, $r_j$ is large, $\theta$ slides up, so the BCM function $r_j(r_j - \theta)$ becomes negative → LTD dominates → weights decrease → neuron fires less. Very fast $\tau_\theta$ creates instability (theta can't settle).

2. **GPU efficiency:** Rate-based BCM is easier to parallelize — no need for incoming CSR or per-spike trace lookups; only rates and thresholds (one value per neuron). STDP requires two CSR representations (incoming + outgoing) and atomicAdd from both pre- and post-spike events — higher memory traffic and more synchronization.

3. **No soft bounds:** Without bounds, LTP would push weights to infinity (or NaN). LTD would push to negative infinity. The weight distribution wouldn't converge — this is why soft bounds are essential.

4. **Thread conflicts:** Two threads i₁ and i₂ conflict when they share a target j and have i₁ → k₁ → j and i₂ → k₂ → j with k₁ = k₂ — impossible since each synapse (k) has a unique pre-neuron. Conflicts in `weights[k]` therefore cannot happen in `bcm_update`. The `atomicAdd` is safe but technically unnecessary here (included for robustness against future kernel modifications).